<a href="https://colab.research.google.com/github/cursuri-inf/IPAP/blob/main/saptamana-03/curs03_demonstratii.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cursul 3. Demonstrațiile

Caietul conține demonstrațiile de la cursul 3, „Fundamente matematice prin cod”, cu rezultatele salvate. Codul este cel din suportul de curs, în aceeași ordine, iar titlurile de mai jos au numerele secțiunilor din suport (§2–§6).

**Cum lucrezi:**

1. Deschizi caietul cu butonul „Open in Colab” de mai sus. Colab avertizează că documentul nu a fost creat de Google: caietul vine din depozitul cursului, așa că alegi „Run anyway”.
2. Înainte să schimbi ceva: File → Save a copy in Drive; lucrezi în copie.
3. Rezultatele salvate se văd și fără rulare. Ca să le obții din nou, rulezi celulele în ordine, de sus în jos, cu Shift+Enter.
4. Apoi schimbi câte un număr (rata de învățare, numărul de epoci, coloanele folosite), rulezi din nou și compari cu ce scrie în suport.
5. Notițele tale: după fiecare demonstrație ai o celulă „Notițele mele” (dublu clic ca s-o editezi). Poți lăsa și comentarii pe marginea din dreapta a oricărei celule: clic dreapta pe celulă, apoi „Add a comment” (sau Ctrl+Alt+M).

Caietul rulează pe procesor; un GPU nu este necesar. Nu se predă și nu se notează.

## 0. Pornire

Bibliotecile, versiunile și datele: cei 342 de pinguini cu toate măsurătorile, ca în cursul 2.

In [1]:
import sys
import torch
from torch import nn
from torch.optim import SGD
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset
import pandas as pd
from sklearn.model_selection import train_test_split

print("Python ", sys.version.split()[0])
print("PyTorch", torch.__version__)

Python  3.13.13
PyTorch 2.11.0+cpu


In [2]:
URL = ("https://raw.githubusercontent.com/mwaskom/"
       "seaborn-data/master/penguins.csv")
masuri = ["bill_length_mm", "bill_depth_mm",
          "flipper_length_mm", "body_mass_g"]
# 342 de pinguini cu toate măsurătorile
df = pd.read_csv(URL).dropna(subset=masuri)
print(df.shape)

(342, 7)


## D1. Tensori din tabelul cu pinguini (§2)

Un tensor este un tablou de numere cu oricâte axe. Forma lui (`shape`) spune câte numere are pe fiecare axă.

In [3]:
masa = torch.tensor(df["body_mass_g"].values,
                    dtype=torch.float32)
print(masa.shape)
print(masa[0])          # primul pinguin

torch.Size([342])
tensor(3750.)


In [4]:
X = torch.tensor(df[masuri[:3]].values,
                 dtype=torch.float32)
print(X.shape)
print(X.dtype)
print(X.mean(dim=0).shape)

torch.Size([342, 3])
torch.float32
torch.Size([3])


In [5]:
# un lot de imagini, ca la laboratorul 2
imagini = torch.zeros(16, 3, 224, 224)
print(imagini.shape, imagini.ndim)
# media maselor este un singur număr: forma goală
print(masa.mean(), masa.mean().shape)

torch.Size([16, 3, 224, 224]) 4
tensor(4201.7544) torch.Size([])


Forma decide ce operații se pot face. La scăderea mediei, vectorul de 3 medii se potrivește singur pe fiecare dintre cele 342 de rânduri (extinderea automată, *broadcasting*). La înmulțirea de matrice, dimensiunile din mijloc trebuie să fie egale.

In [6]:
# [342, 3] − [3] → [342, 3]
Z = (X - X.mean(dim=0)) / X.std(dim=0)
print(Z.shape, Z.std(dim=0))

torch.Size([342, 3]) tensor([1., 1., 1.])


In [7]:
w = torch.tensor([[0.5], [0.2]])   # [2, 1]
try:
    X @ w            # [342, 3] @ [2, 1]
except RuntimeError as e:
    print(e)
w = torch.tensor([[0.5], [0.2], [0.1]])   # [3, 1]
print((X @ w).shape)   # [342, 3] @ [3, 1]

mat1 and mat2 shapes cannot be multiplied (342x3 and 2x1)
torch.Size([342, 1])


Pe un runtime cu GPU (Runtime → Change runtime type → T4 GPU), tensorul se mută pe placa video cu `.to`. Celula de mai jos afișează `cpu` pe procesor și `cuda:0` pe GPU.

In [8]:
gpu = torch.cuda.is_available()
X = X.to("cuda" if gpu else "cpu")
print(X.device)

cpu


### Notițele mele

…

## D2. Derivata și gradientul, pe cei cinci pinguini (§3)

Modelul din cursul 2: masa = a × lungimea aripii, cu un singur parametru, `a`. Pentru gradient folosim pierderea pătratică medie (`mse`); eroarea absolută medie (`mae`) rămâne metrica pe care o citim, în grame.

In [9]:
# cinci pinguini: aripa (mm) și masa (g)
x = torch.tensor([181., 186., 195., 193., 190.])
y = torch.tensor([3750., 3800., 3250., 3450.,
                  3650.])

def mse(a):   # pierderea pătratică medie
    return ((a * x - y) ** 2).mean()

def mae(a):   # eroarea absolută medie
    return (a * x - y).abs().mean()

def panta(f, a, h=0.01):   # panta lui f în a
    return (f(a+h) - f(a-h)) / (2*h)

print(round(mse(10.0).item()),
      round(panta(mse, 10.0).item()))

2918020 -636538


Panta erorii absolute rămâne aceeași departe de minim; panta pierderii pătratice scade pe măsură ce ne apropiem de el.

In [10]:
for a in (10.0, 14.0, 18.0, 18.9):
    print(a, round(panta(mae, a).item()),
          round(panta(mse, a).item()))

10.0 -189 -636538
14.0 -189 -350575
18.0 -34 -64602
18.9 -34 -254


Coborârea pe gradient: la fiecare pas, `a` se mută în sens opus pantei, cu un pas proporțional cu ea.

In [11]:
a, rata = 10.0, 0.00001
for pas in range(4):
    print(pas, round(a, 2), round(mae(a).item()))
    a = a - rata * panta(mse, a).item()

0 10.0 1690
1 16.37 487
2 18.18 286
3 18.7 268


PyTorch calculează panta singur: `requires_grad=True` îi cere să țină minte calculul, iar `backward()` pune panta în `a.grad`.

In [12]:
a = torch.tensor(10.0, requires_grad=True)
mse(a).backward()
print(a.grad)
for pas in range(3):
    # pasul se face în afara calculului urmărit
    with torch.no_grad():
        a -= 0.00001 * a.grad
        a.grad.zero_()   # ștergem gradientul vechi
    mse(a).backward()
    print(pas + 1, round(a.item(), 2))

tensor(-636536.)
1 16.37
2 18.18
3 18.7


### Notițele mele

…

## D3. Rata de învățare (§4)

Aceeași coborâre, cu trei rate de învățare, pornind tot de la a = 10.

In [13]:
def coboara(rata, pasi=4, a=10.0):
    drum = [a]
    for _ in range(pasi):
        a = a - rata * panta(mse, a).item()
        drum.append(round(a, 2))
    return drum

for rata in (0.00001, 0.00002, 0.00003):
    print(rata, coboara(rata))

1e-05 [10.0, 16.37, 18.18, 18.7, 18.84]
2e-05 [10.0, 22.73, 17.26, 19.61, 18.6]
3e-05 [10.0, 29.1, 7.24, 32.26, 3.62]


In [14]:
for rata in (0.000001, 0.00001, 0.00003):
    a = 10.0
    for pas in range(20):
        a = a - rata * panta(mse, a).item()
    print(rata, round(a, 2), round(mse(a).item()))

1e-06 16.88 230111
1e-05 18.9 84306
3e-05 -114.04 631900160


### Notițele mele

…

## D4. Regresia liniară cu gradient (§5)

Aceeași întrebare ca la sfârșitul cursului 2: cât cântărește un pinguin, după lungimea aripii? Aceeași împărțire, 80% antrenare și 20% test, cu `random_state=0`.

In [15]:
Xr_ant, Xr_test, yr_ant, yr_test = train_test_split(
    df[["flipper_length_mm"]], df["body_mass_g"],
    test_size=0.2, random_state=0)

def coloana(t):   # din pandas: tensor cu o coloană
    v = torch.tensor(t.values).float()
    return v.reshape(-1, 1)

x_ant, y_ant = coloana(Xr_ant), coloana(yr_ant)
x_test, y_test = coloana(Xr_test), coloana(yr_test)
print(x_ant.shape, x_test.shape)

torch.Size([273, 1]) torch.Size([69, 1])


In [16]:
# media și abaterea: doar din setul de antrenare
medie, abatere = x_ant.mean(), x_ant.std()
z_ant = (x_ant - medie) / abatere
z_test = (x_test - medie) / abatere
print(round(medie.item(), 1),
      round(abatere.item(), 1))

201.1 14.4


In [17]:
torch.manual_seed(0)
model = nn.Linear(1, 1)          # masa = w × z + b
pierdere_f = nn.MSELoss()
opt = SGD(model.parameters(), lr=0.02)
loturi = DataLoader(TensorDataset(z_ant, y_ant),
                    batch_size=32, shuffle=True)

for epoca in range(50):
    for zb, yb in loturi:
        predictie = model(zb)
        pierdere = pierdere_f(predictie, yb)
        opt.zero_grad()
        pierdere.backward()
        opt.step()
# loturi pe epocă și pași în total
print(len(loturi), len(loturi) * 50)

9 450


In [18]:
# înapoi la grame pe milimetru: masa ≈ w × aripa + b
w = model.weight.item() / abatere.item()
b = model.bias.item() - w * medie.item()
with torch.no_grad():
    eroare = (model(z_test) - y_test).abs().mean()
print(round(w, 1), round(b), round(eroare.item()))
print(round((yr_ant.mean() - yr_test).abs().mean()))

49.4 -5727 314
619


Fără standardizare, cu aripa în milimetri: rata trebuie să fie foarte mică, altfel pierderea explodează, iar cu o rată atât de mică termenul liber `b` aproape nu se mișcă. Încearcă și rata 0.00003.

In [19]:
torch.manual_seed(0)
fara = nn.Linear(1, 1)
opt_f = SGD(fara.parameters(), lr=0.00001)
for epoca in range(1000):
    p = pierdere_f(fara(x_ant), y_ant)
    opt_f.zero_grad()
    p.backward()
    opt_f.step()
with torch.no_grad():
    eroare = (fara(x_test) - y_test).abs().mean()
print(round(fara.weight.item(), 1),
      round(fara.bias.item(), 2),
      round(eroare.item()))

21.1 0.06 456


### Notițele mele

…

## D5. Regresia logistică și softmax (§6)

Împărțirea pentru clasificare din cursul 2: 70% antrenare, 15% validare, 15% test, stratificat pe specii.

In [20]:
X_rest, X_test, s_rest, s_test = train_test_split(
    df[masuri], df["species"], test_size=0.15,
    stratify=df["species"], random_state=0)
X_ant, X_val, s_ant, s_val = train_test_split(
    X_rest, s_rest, test_size=0.15 / 0.85,
    stratify=s_rest, random_state=0)
print(len(X_ant), len(X_val), len(X_test))

238 52 52


In [21]:
scoruri = torch.tensor([-2.0, 0.0, 1.0, 2.0])
print(torch.sigmoid(scoruri))
p = torch.tensor([0.99, 0.9, 0.5, 0.1, 0.01])
print(-torch.log(p))   # pierderile din cursul 2

tensor([0.1192, 0.5000, 0.7311, 0.8808])
tensor([0.0101, 0.1054, 0.6931, 2.3026, 4.6052])


In [22]:
def intrari(t, col):   # standardizate pe antrenare
    m, s = X_ant[col].mean(), X_ant[col].std()
    z = (t[col] - m) / s
    return torch.tensor(z.values).float()

def gentoo(s):   # 1 pentru Gentoo, altfel 0
    t = torch.tensor((s == "Gentoo").values)
    return t.float().reshape(-1, 1)

col = ["bill_depth_mm", "flipper_length_mm"]
z_ant = intrari(X_ant, col)
z_val = intrari(X_val, col)
t_ant, t_val = gentoo(s_ant), gentoo(s_val)
print(z_ant.shape, round(t_val.mean().item(), 3))

torch.Size([238, 2]) 0.365


In [23]:
torch.manual_seed(0)
# scorul = w1 × cioc + w2 × aripa + b
neuron = nn.Linear(2, 1)
pierdere_f = nn.BCEWithLogitsLoss()
opt = SGD(neuron.parameters(), lr=0.5)
for epoca in range(200):
    pierdere = pierdere_f(neuron(z_ant), t_ant)
    opt.zero_grad()
    pierdere.backward()
    opt.step()
with torch.no_grad():
    prob = torch.sigmoid(neuron(z_val))
corect = (prob > 0.5).float() == t_val
print(round(pierdere.item(), 3),
      round(corect.float().mean().item(), 3))

0.017 1.0


In [24]:
# scorurile pentru Adelie, Chinstrap, Gentoo
scoruri = torch.tensor([1.95, 0.7, 0.0])
p = torch.softmax(scoruri, dim=0)
print(p.round(decimals=2))
print(round(-torch.log(p[0]).item(), 2))

tensor([0.7000, 0.2000, 0.1000])
0.36


In [25]:
specii = ["Adelie", "Chinstrap", "Gentoo"]
z3_ant = intrari(X_ant, masuri[:3])
z3_val = intrari(X_val, masuri[:3])
c_ant = torch.tensor(s_ant.map(specii.index).values)
c_val = torch.tensor(s_val.map(specii.index).values)

torch.manual_seed(0)
strat = nn.Linear(3, 3)   # un scor pe specie
pierdere_f = nn.CrossEntropyLoss()
opt = SGD(strat.parameters(), lr=0.5)
for epoca in range(300):
    pierdere = pierdere_f(strat(z3_ant), c_ant)
    opt.zero_grad()
    pierdere.backward()
    opt.step()
with torch.no_grad():
    alese = strat(z3_val).argmax(dim=1)
corect = alese == c_val
print(round(corect.float().mean().item(), 3))

0.981


### Notițele mele

…

## Mai departe

La laboratorul 3 antrenezi singur ambele modele, cu rândurile buclei de completat. Caietele de rampă 2 („Algebră liniară și gradient prin cod”) și 3 („Probabilități și entropie încrucișată”) repetă aceleași idei, cu exerciții care se verifică singure.